# Day 1 — The Data Detective 🔍

> **Mission briefing:** Welcome to the Lab, recruit. Before we let you anywhere near an AI model, you have to pass the entrance exam — and it isn't about code. Every powerful AI system is built on top of *data*, and a researcher who can't **read** their data ships broken models. This morning a crate of penguin measurements arrived from a field station in Antarctica. Interrogate it.

**Today you will:**
- Load a real dataset with **pandas** and size it up (`.head`, `.info`, `.describe`)
- Hunt down **missing values** and clean the data
- **Filter, group, and count** to answer real questions
- **Plot** the data — and spot a pattern a machine could learn tomorrow

*No neural networks today. Just you, the data, and a magnifying glass. Every researcher here will tell you the same thing: **AI is 80% understanding your data, and 20% everything else.***

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║ SETUP — run me first! (identical in every Lab notebook)           ║
# ╚═══════════════════════════════════════════════════════════════════╝
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SMOKE_TEST", "0") == "1"   # tiny fast run for automated tests

REPO_URL = "https://github.com/CHANGE-ME/ai-studio-internship"  # instructor: set once

if IN_COLAB:
    if not pathlib.Path("/content/ai-studio-internship").exists():
        !git clone -q {REPO_URL} /content/ai-studio-internship
    %cd /content/ai-studio-internship/notebooks/day01

HERE = pathlib.Path.cwd()
REPO = HERE.parents[1]                       # .../notebooks/dayNN -> repo root
DATA_DIR = pathlib.Path(os.environ.get("COURSE_DATA_DIR", REPO / "data"))
SAMPLES_DIR = REPO / "data_samples"          # small datasets shipped with the repo
MODELS_DIR = REPO / "ai_studio" / "models"   # where your Studio modules unlock
for p in (REPO / "data", MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

import random
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print(f"✅ Setup done | Colab: {IN_COLAB} | Smoke test: {SMOKE}")
print(f"   data: {DATA_DIR}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)   # never hide a column when we print
print("Detective toolkit loaded 🕵️")

## 🐧 Meet the suspects

Our data comes from **Palmer Station, Antarctica**, where researchers measured hundreds of penguins from **three species**:

| Species | How to spot it |
|---|---|
| **Adelie** | The classic tuxedo penguin, found all over the peninsula |
| **Chinstrap** | Named for the thin black line under its chin |
| **Gentoo** | The big one, with a bright orange bill |

For each penguin they recorded four **body measurements**, plus which **island** it lived on and its **sex**:

- `bill_length_mm`, `bill_depth_mm` — the beak
- `flipper_length_mm` — the "wing"
- `body_mass_g` — weight in grams

One **row** is one penguin. One **column** is one thing we measured. Rows are examples, columns are features — this is *exactly* how almost all machine-learning data is organized. Get comfortable with the shape.

### Step 1 — Load the evidence

`pandas` is the tool every data scientist reaches for first. It reads a spreadsheet-like file into a **DataFrame** — think of it as a programmable Excel sheet. `.head()` shows the first few rows so we can eyeball what we are dealing with.

In [ ]:
penguins = pd.read_csv(SAMPLES_DIR / "penguins.csv")
print(f"Loaded {len(penguins)} penguins, {penguins.shape[1]} columns each.")
penguins.head()

### Step 2 — Size it up

Two commands you will run on *every* new dataset for the rest of your career:

- **`.info()`** — how many rows, the type of each column, and how many values are actually *present* (non-null). The place to spot **missing** data.
- **`.describe()`** — count, mean, min, max, and quartiles for every number column. The place to spot **weird** values (a penguin weighing 0 g? a typo?).

In [ ]:
penguins.info()

In [ ]:
penguins.describe()

### 🧪 Exercise 1 — Count the gaps

Real data is messy. Some penguins are missing measurements — maybe a pen ran dry, maybe a bird wriggled free before the scale settled.

- Compute how many values are **missing** in each column and store it in `missing_per_column`.
- **Expected:** a couple of missing body measurements, and about a dozen missing `sex` values.

In [ ]:
# How many values are missing in each column?
# ==================== YOUR CODE HERE ====================
### HINT: penguins.isna() marks every cell True (missing) or False
### HINT: chain .sum() on that to count the missing values per column
...
print(missing_per_column)

assert missing_per_column.sum() == 19, "Expected 19 missing values in total across all columns."

### Step 3 — Clean it up

For today's investigation we want only penguins with a **complete** record. `dropna()` removes any row that has at least one missing value. We keep the original around and work on a clean copy called `df`.

> ⚠️ Dropping rows is the quick fix, not always the *right* one — throwing away data can bias your results. Later in the Lab you will learn gentler repairs (filling a gap with a sensible guess). For 11 rows out of 344, dropping is fine.

In [ ]:
df = penguins.dropna().reset_index(drop=True)
print(f"Before: {len(penguins)} penguins")
print(f"After : {len(df)} penguins   ({len(penguins) - len(df)} dropped)")
assert len(df) == 333, "A clean dataset should have 333 complete records."
df.head()

## Ask questions, get answers

A DataFrame is not just for looking at — it is for **interrogating**. Three moves answer almost any question about a dataset:

- **Filter** — keep only the rows that match a condition
- **Group** — split rows into buckets (e.g. by species) and summarize each bucket
- **Count** — tally how often each value appears

Let's use all three.

### 🧪 Exercise 2 — The heavyweights

Find every penguin heavier than **5000 grams** (that's 5 kg — a chunky bird).

- Build a filtered DataFrame called `heavy`.
- **Expected:** around 60 penguins — and if you peek at their species, they will almost all be the *same* one. Remember which; it comes up again.

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: a condition like df["body_mass_g"] > 5000 gives True/False for every row
### HINT: put that condition inside df[ ... ] to keep only the True rows
...
print(f"{len(heavy)} penguins over 5000 g")
print(heavy["species"].value_counts())

assert len(heavy) > 40, "That is fewer heavyweights than expected — check your condition."

### 🧪 Exercise 3 — The average penguin, by species

Split the penguins by species and compute the **mean** of every numeric column for each group.

- Produce a table called `species_means`.
- **Hint:** `.groupby("species")` makes the buckets; `.mean(numeric_only=True)` averages the numbers (the `numeric_only=True` skips text columns like `island`).
- **Expected:** one row per species, with Gentoo standing out as the biggest on nearly every measurement.

In [ ]:
# ==================== YOUR CODE HERE ====================
...
species_means

### 🧪 Exercise 4 — Who lives where?

Count how many penguins were recorded on each **island**.

- Store the result in `island_counts`.
- **Expected:** three islands (Biscoe, Dream, Torgersen), with Biscoe hosting the most.

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: .value_counts() on a single column tallies each distinct value
...
print(island_counts)

## See it to believe it

Tables are precise but slow to read. A good plot shows a pattern in a glance. Start with a **histogram**: it slices a numeric column into bins and shows how many penguins land in each — the *shape* of the data.

Here is one we made for you: body mass, drawn separately for each species so you can compare. Run it.

In [ ]:
plt.figure(figsize=(8, 5))
for species, group in df.groupby("species"):
    plt.hist(group["body_mass_g"], bins=20, alpha=0.6, label=species)
plt.xlabel("Body mass (g)")
plt.ylabel("Number of penguins")
plt.title("Body mass by species")
plt.legend()
plt.show()

See how **Gentoo** (heavier) sits clearly to the right, while **Adelie** and **Chinstrap** overlap almost completely? Body mass alone can pick out a Gentoo — but it can't tell an Adelie from a Chinstrap. We need a smarter view.

### 🧪 Exercise 5 — The scatter plot that changes everything

A **scatter plot** puts one measurement on the x-axis and another on the y-axis, one dot per penguin. The magic move: **color each dot by species**. If the species land in separate regions, we have found structure a machine could learn.

Plot **bill length** (x) against **bill depth** (y), one color per species.

- Loop over the species; for each one, call `plt.scatter(...)` on just that group's rows with a `label=`.
- The scaffold handles the axis labels and legend — you write the scatter loop.
- **Expected:** three fairly distinct blobs of color.

In [ ]:
plt.figure(figsize=(8, 6))
# ==================== YOUR CODE HERE ====================
### HINT: for species, group in df.groupby("species"):
### HINT:     plt.scatter(group["bill_length_mm"], group["bill_depth_mm"], label=species, alpha=0.7)
...
plt.xlabel("Bill length (mm)")
plt.ylabel("Bill depth (mm)")
plt.title("Bill shape separates the species")
plt.legend()
plt.show()

### 🔍 The revelation

Look at that. Each species forms its **own cluster** — a region of the plot where its penguins live. You could almost draw the dividing lines with a ruler.

That is the whole idea behind tomorrow's mission. **If *you* can see the boundary, a machine can *learn* it.** Hand a computer these dots and their colors, and it can work out the dividing lines on its own — then color in a brand-new, unlabeled dot for you. That's called **classification**, and it's the first AI superpower you'll build. 🐧

> **Why bill *length* and *depth*?** Because together they capture *shape*, not just size. Size (mass, flipper) mostly separates Gentoo from the rest; shape is what finally splits Adelie from Chinstrap.

### 🧪 Exercise 6 — What moves together?

When one measurement goes up, do the others go up too? **Correlation** measures exactly that, from **+1** (perfect lockstep), through **0** (unrelated), to **−1** (perfect opposites).

- Compute the correlation matrix of the four numeric columns and store it in `corr`.
- We'll draw it as a **heatmap** so the strong pairs jump out.
- **Hint:** select the four columns, then call `.corr()`.

In [ ]:
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
# ==================== YOUR CODE HERE ====================
...

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation between measurements")
plt.show()
corr

**Read the map.** The brightest off-diagonal square is **flipper length ↔ body mass** at about **+0.87** — long-flippered penguins are heavy penguins. That makes sense: both grow with the overall size of the bird.

Notice that `bill_length` and `bill_depth` are only weakly (even negatively) correlated — which is *exactly* why they made such a good scatter plot. Two measurements that *don't* move together carry **independent** information, and independent information is what pulls the species apart.

## 🗒️ The Detective's Report

Time to prove you can work solo.

### 🧪 Exercise 7 — Close the case

Answer three questions with **one line of pandas each** — no plots, just the value. Everything you need you have already used above.

1. Which species is **heaviest on average**?
2. Which **island** hosts the most **Gentoo** penguins?
3. What is the **mean flipper length** of **Chinstrap** penguins?

In [ ]:
# 1) Heaviest species on average
# ==================== YOUR CODE HERE ====================
### HINT: group by species, take the mean body_mass_g, then .idxmax() names the winner
...

# 2) Island with the most Gentoo penguins
# ==================== YOUR CODE HERE ====================
### HINT: keep only Gentoo rows first, then value_counts the island column, then .idxmax()
...

# 3) Mean flipper length of Chinstrap penguins
# ==================== YOUR CODE HERE ====================
...

print(f"1. Heaviest species on average : {heaviest_species}")
print(f"2. Island with most Gentoo     : {top_gentoo_island}")
print(f"3. Chinstrap mean flipper (mm) : {chinstrap_flipper:.1f}")

assert heaviest_species == "Gentoo"
assert top_gentoo_island == "Biscoe"
assert round(chinstrap_flipper, 1) == 195.8
print("\n✅ Case closed — all three answers check out.")

## 🏁 Checkpoint

Case closed, Detective. You just did what every AI project starts with — long before a single model is trained:

- ✅ Loaded and inspected a real dataset (`.head`, `.info`, `.describe`)
- ✅ Found and removed missing data
- ✅ Filtered, grouped, and counted to answer real questions
- ✅ Visualized the data — and spotted **species clusters** with your own eyes
- ✅ Measured which features carry independent information

You proved the Lab's motto: **understand your data before you model it.**

**Next up — Notebook 1b: "Meet the Machines." 🤖** You've earned a reward: a hands-on tour of four finished AI systems — vision, mood-reading, story-writing, and a sorting hat. Play with them today; you will *build* every one over the next two weeks.

> **Tomorrow (Day 2):** those penguin clusters you found? You'll hand them to a machine, train it to identify a brand-new penguin on its own, and unlock the very first tab of your **AI Studio**. 🔓